<a href="https://colab.research.google.com/github/Aravind-9092/CropSynth/blob/main/NLP_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
# ════════════════════════════════════════════════════════════════
# QUANTUM-INSPIRED WSD — FINAL COMPLETE PIPELINE
# ════════════════════════════════════════════════════════════════

# ── STEP 1: INSTALL ──────────────────────────────────────────────
!pip install torch kaggle nltk -q

# ── STEP 2: IMPORTS ──────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.optim as optim
from collections import Counter, defaultdict
import xml.etree.ElementTree as ET
import os
import nltk
import random
from torch.utils.data import DataLoader, TensorDataset

nltk.download('wordnet')
nltk.download('omw-1.4')
from nltk.corpus import wordnet as wn

# ── STEP 3: KAGGLE SETUP & DOWNLOAD ─────────────────────────────
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d nltkdata/semcor-corpus
!unzip -o semcor-corpus.zip

# ── STEP 4: CONFIG ───────────────────────────────────────────────
MAX_LEN    = 32
D_MODEL    = 128
NUM_SENSES = 4
N_HEADS    = 4
N_LAYERS   = 2
BATCH_SIZE = 32
EPOCHS     = 100
LR         = 2e-4
SEED       = 42

random.seed(SEED)
torch.manual_seed(SEED)

AMBIGUOUS_WORDS = {
    "bank", "bat", "light", "plant", "bass", "bow", "bark",
    "match", "well", "spring", "rock", "fly", "ring", "watch",
    "run", "play", "hard", "left", "right", "interest", "fair",
    "board", "head", "arm", "face", "hand", "line", "table",
    "saw", "bear", "mine", "pitch", "file", "pool", "post"
}

# ── STEP 5: PARSE SEMCOR (no limit — use ALL data) ───────────────
def parse_semcor_wsd(path):
    data = []
    for root_dir, _, files in os.walk(path):
        for file in sorted(files):
            if not file.endswith(".xml"):
                continue
            tree = ET.parse(os.path.join(root_dir, file))
            root = tree.getroot()
            for s_tag in root.iter('s'):
                words      = []
                candidates = []
                for i, wf in enumerate(s_tag):
                    word = wf.text
                    if word is None:
                        continue
                    word = word.lower().strip()
                    words.append(word)
                    lemma = wf.get('lemma', '').lower()
                    lexsn = wf.get('lexsn', '')
                    if lemma in AMBIGUOUS_WORDS and lexsn:
                        candidates.append((i, lemma, f"{lemma}.{lexsn}"))
                if len(words) < 3:
                    continue
                for (idx, lemma, sense_key) in candidates:
                    data.append({
                        "words":      words,
                        "target_idx": idx,
                        "lemma":      lemma,
                        "sense":      sense_key
                    })
    return data

data = parse_semcor_wsd("semcor/")
print(f"Total samples loaded: {len(data)}")

# ── STEP 6: BUILD VOCABULARIES ───────────────────────────────────
def build_word_vocab(data):
    counter = Counter()
    for d in data:
        counter.update(d["words"])
    vocab = {"<PAD>": 0, "<UNK>": 1}
    for word, _ in counter.most_common():
        vocab[word] = len(vocab)
    return vocab

def build_sense_vocab_per_lemma(data):
    lemma_senses = defaultdict(set)
    for d in data:
        lemma_senses[d["lemma"]].add(d["sense"])
    return {
        lemma: {s: i for i, s in enumerate(sorted(senses))}
        for lemma, senses in lemma_senses.items()
    }

word_vocab        = build_word_vocab(data)
lemma_sense_vocab = build_sense_vocab_per_lemma(data)

# ── STEP 7: FILTER TO MULTI-SENSE LEMMAS ─────────────────────────
multi_sense_lemmas = {
    lemma for lemma, sv in lemma_sense_vocab.items() if len(sv) >= 2
}
data = [d for d in data if d["lemma"] in multi_sense_lemmas]

lemma_list = sorted(multi_sense_lemmas)
max_senses = max(len(lemma_sense_vocab[l]) for l in lemma_list)

print(f"Samples after filtering : {len(data)}")
print(f"Ambiguous lemmas        : {lemma_list}")
print(f"Max senses any lemma    : {max_senses}")

# Print per-lemma sample counts so we can see the imbalance
print("\nSamples per lemma:")
lemma_counts = Counter(d["lemma"] for d in data)
for lemma in lemma_list:
    n_senses = len(lemma_sense_vocab[lemma])
    print(f"  {lemma:<12} {lemma_counts[lemma]:4d} samples,  {n_senses} senses")

# ── STEP 8: TRAIN / VAL SPLIT ────────────────────────────────────
# Split per-lemma to ensure every sense appears in both sets
train_data, val_data = [], []
for lemma in lemma_list:
    lemma_samples = [d for d in data if d["lemma"] == lemma]
    random.shuffle(lemma_samples)
    cut = max(1, int(len(lemma_samples) * 0.85))
    train_data.extend(lemma_samples[:cut])
    val_data.extend(lemma_samples[cut:])

random.shuffle(train_data)
print(f"\nTrain: {len(train_data)}  |  Val: {len(val_data)}")

# ── STEP 9: DATA AUGMENTATION ────────────────────────────────────
# Word dropout: randomly replace non-target words with <UNK>
# This forces the model to generalize beyond exact word patterns

def augment_sample(d, word_vocab, dropout_prob=0.15):
    """Return a copy with random non-target words replaced by <UNK>."""
    words   = d["words"][:]
    tgt_idx = d["target_idx"]
    new_words = []
    for i, w in enumerate(words):
        if i == tgt_idx:
            new_words.append(w)           # never drop the target word
        elif random.random() < dropout_prob:
            new_words.append("<UNK>")     # drop context word
        else:
            new_words.append(w)
    return {**d, "words": new_words}

# ── STEP 10: ENCODE DATA ─────────────────────────────────────────
def encode_sample(d, word_vocab, max_len=MAX_LEN):
    words   = d["words"]
    tgt_idx = d["target_idx"]
    lemma   = d["lemma"]

    ids = [word_vocab.get(w, 1) for w in words]
    if len(ids) > max_len:
        start   = max(0, tgt_idx - max_len // 2)
        ids     = ids[start : start + max_len]
        tgt_idx = tgt_idx - start
    tgt_idx = min(tgt_idx, len(ids) - 1)
    ids     = ids + [0] * (max_len - len(ids))
    return ids, tgt_idx

def encode_dataset(data, word_vocab, lemma_sense_vocab, augment=False, max_len=MAX_LEN):
    lemma_to_id = {l: i for i, l in enumerate(lemma_list)}
    X, target_pos, Y, lemma_ids = [], [], [], []

    for d in data:
        if augment:
            d = augment_sample(d, word_vocab)

        lemma     = d["lemma"]
        sense_key = d["sense"]
        if sense_key not in lemma_sense_vocab[lemma]:
            continue

        ids, tgt_idx = encode_sample(d, word_vocab, max_len)
        X.append(ids)
        target_pos.append(tgt_idx)
        Y.append(lemma_sense_vocab[lemma][sense_key])
        lemma_ids.append(lemma_to_id[lemma])

    return (
        torch.tensor(X,          dtype=torch.long),
        torch.tensor(target_pos, dtype=torch.long),
        torch.tensor(Y,          dtype=torch.long),
        torch.tensor(lemma_ids,  dtype=torch.long),
    )

Xv, pv, Yv, Lv = encode_dataset(val_data, word_vocab, lemma_sense_vocab, augment=False)
val_loader = DataLoader(TensorDataset(Xv, pv, Yv, Lv), batch_size=BATCH_SIZE)

print(f"Val tensor: {Xv.shape}")

# ── STEP 11: SENSE DEFINITIONS ───────────────────────────────────
def build_sense_definitions():
    defs = {}
    for lemma, sv in lemma_sense_vocab.items():
        if lemma not in multi_sense_lemmas:
            continue
        for sense_key in sv:
            try:
                parts  = sense_key.split('.')
                wn_key = parts[0] + '%' + parts[1]
                synset = wn.lemma_from_key(wn_key).synset()
                defs[sense_key] = synset.definition()
            except Exception:
                try:
                    pos_code = int(sense_key.split('.')[1].split(':')[0])
                    pos_map  = {1: wn.NOUN, 2: wn.VERB,
                                3: wn.ADJ,  4: wn.ADV, 5: wn.ADJ}
                    lex_id   = int(sense_key.split(':')[2])
                    synsets  = wn.synsets(lemma, pos=pos_map.get(pos_code, wn.NOUN))
                    idx      = min(lex_id, len(synsets) - 1)
                    defs[sense_key] = synsets[idx].definition() if synsets else sense_key
                except Exception:
                    defs[sense_key] = sense_key
    return defs

print("Building sense definitions...")
sense_definitions = build_sense_definitions()

print("\nBank senses:")
for sk in sorted(k for k in sense_definitions if k.startswith("bank")):
    print(f"  {sk:<25} {sense_definitions[sk][:65]}")

# ── STEP 12: MODEL ───────────────────────────────────────────────

class ContextAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.q_proj   = nn.Linear(d_model, d_model)
        self.k_proj   = nn.Linear(d_model, d_model)
        self.v_proj   = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.norm     = nn.LayerNorm(d_model)
        self.n_heads  = n_heads
        self.d_head   = d_model // n_heads
        self.scale    = self.d_head ** -0.5

    def forward(self, ctx, target_pos, pad_mask=None):
        B, L, D = ctx.shape
        H, Dh   = self.n_heads, self.d_head
        idx     = target_pos.view(B, 1, 1).expand(B, 1, D)
        tgt     = ctx.gather(1, idx)

        Q = self.q_proj(tgt).view(B, 1, H, Dh).transpose(1, 2)
        K = self.k_proj(ctx).view(B, L, H, Dh).transpose(1, 2)
        V = self.v_proj(ctx).view(B, L, H, Dh).transpose(1, 2)

        scores = torch.matmul(Q, K.transpose(-2, -1)) * self.scale
        if pad_mask is not None:
            scores = scores.masked_fill(pad_mask.unsqueeze(1).unsqueeze(2), -1e4)

        weights = torch.softmax(scores, dim=-1)
        out     = torch.matmul(weights, V)
        out     = out.transpose(1, 2).contiguous().view(B, 1, D)
        out     = self.out_proj(out).squeeze(1)
        return self.norm(out + tgt.squeeze(1))


class QuantumCollapse(nn.Module):
    def __init__(self, d_model, num_senses):
        super().__init__()
        self.num_senses = num_senses
        self.sense_proj = nn.Linear(d_model, d_model * num_senses)
        self.phase_proj = nn.Linear(d_model, num_senses)
        self.norm       = nn.LayerNorm(d_model)

    def forward(self, x):
        B, D    = x.shape
        senses  = self.sense_proj(x).view(B, self.num_senses, D)
        phases  = self.phase_proj(x)
        mag     = torch.cos(phases) ** 2 + torch.sin(phases) ** 2
        weights = torch.softmax(mag, dim=-1)
        return self.norm(torch.einsum('bs,bsd->bd', weights, senses))


class WSDModel(nn.Module):
    def __init__(self, vocab_size, d_model=D_MODEL, num_senses=NUM_SENSES,
                 n_heads=N_HEADS, n_layers=N_LAYERS, max_seq=MAX_LEN):
        super().__init__()
        self.token_embed = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.pos_embed   = nn.Embedding(max_seq, d_model)
        self.embed_norm  = nn.LayerNorm(d_model)
        self.embed_drop  = nn.Dropout(0.2)      # increased from 0.1

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads,
            dim_feedforward=d_model * 2,
            dropout=0.2,                         # increased from 0.1
            batch_first=True, norm_first=True
        )
        self.encoder  = nn.TransformerEncoder(enc_layer, num_layers=n_layers)
        self.ctx_attn = ContextAttention(d_model, n_heads)
        self.quantum  = QuantumCollapse(d_model, num_senses)
        self.drop     = nn.Dropout(0.3)          # increased from 0.2

        self.heads = nn.ModuleDict({
            lemma: nn.Linear(d_model, len(lemma_sense_vocab[lemma]))
            for lemma in multi_sense_lemmas
        })

    def forward(self, x, target_pos, lemma_name):
        B, L  = x.shape
        pos   = torch.arange(L, device=x.device).unsqueeze(0)
        pad   = (x == 0)
        emb   = self.token_embed(x) + self.pos_embed(pos)
        emb   = self.embed_drop(self.embed_norm(emb))
        ctx   = self.encoder(emb, src_key_padding_mask=pad)
        tgt   = self.ctx_attn(ctx, target_pos, pad)
        q     = self.drop(self.quantum(tgt))
        return self.heads[lemma_name](q)

# ── STEP 13: TRAINING ────────────────────────────────────────────
model     = WSDModel(vocab_size=len(word_vocab))
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=5e-3)  # stronger decay
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
criterion = nn.CrossEntropyLoss(label_smoothing=0.15)  # stronger smoothing

print(f"Model parameters : {sum(p.numel() for p in model.parameters()):,}")
print(f"Train samples    : {len(train_data)}")
print(f"Val samples      : {len(val_data)}\n")

best_val_acc  = 0.0
patience      = 0
EARLY_STOP    = 20     # stop if val doesn't improve for 20 epochs

for epoch in range(1, EPOCHS + 1):
    # Re-encode train with fresh augmentation each epoch
    Xtr, ptr, Ytr, Ltr = encode_dataset(
        train_data, word_vocab, lemma_sense_vocab, augment=True
    )
    train_loader = DataLoader(
        TensorDataset(Xtr, ptr, Ytr, Ltr),
        batch_size=BATCH_SIZE, shuffle=True
    )

    # ── train ──
    model.train()
    tr_loss = tr_correct = tr_total = 0

    for x_b, pos_b, y_b, lid_b in train_loader:
        optimizer.zero_grad()

        lemma_groups = defaultdict(list)
        for i in range(x_b.size(0)):
            lemma_groups[lemma_list[lid_b[i].item()]].append(i)

        batch_loss = torch.tensor(0.0, device=x_b.device)
        correct    = 0
        for lemma, indices in lemma_groups.items():
            idx_t  = torch.tensor(indices)
            logits = model(x_b[idx_t], pos_b[idx_t], lemma)
            labels = y_b[idx_t]
            loss   = criterion(logits, labels)
            batch_loss = batch_loss + loss * len(indices)
            correct   += (logits.argmax(-1) == labels).sum().item()

        batch_loss = batch_loss / x_b.size(0)
        batch_loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        tr_loss    += batch_loss.item()
        tr_correct += correct
        tr_total   += x_b.size(0)

    scheduler.step()

    # ── validate ──
    model.eval()
    val_correct = val_total = 0
    with torch.no_grad():
        for x_b, pos_b, y_b, lid_b in val_loader:
            for i in range(x_b.size(0)):
                lemma  = lemma_list[lid_b[i].item()]
                logits = model(x_b[i:i+1], pos_b[i:i+1], lemma)
                val_correct += int(logits.argmax(-1).item() == y_b[i].item())
                val_total   += 1

    tr_acc  = 100.0 * tr_correct  / tr_total
    val_acc = 100.0 * val_correct / val_total

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience     = 0
        torch.save(model.state_dict(), "best_wsd.pt")
    else:
        patience += 1

    if epoch % 10 == 0 or epoch == 1:
        print(f"Epoch {epoch:3d}  |  Loss: {tr_loss:.3f}  |  "
              f"Train: {tr_acc:.1f}%  |  Val: {val_acc:.1f}%  |  "
              f"Best: {best_val_acc:.1f}%  |  Patience: {patience}/{EARLY_STOP}")

    if patience >= EARLY_STOP:
        print(f"\nEarly stop at epoch {epoch}  (best val: {best_val_acc:.1f}%)")
        break

model.load_state_dict(torch.load("best_wsd.pt"))
print(f"\nLoaded best model — val acc: {best_val_acc:.1f}%")

# ── STEP 14: DETAILED EVAL PER LEMMA ─────────────────────────────
print("\nPer-lemma validation accuracy:")
model.eval()
lemma_correct = defaultdict(int)
lemma_total   = defaultdict(int)

with torch.no_grad():
    for x_b, pos_b, y_b, lid_b in val_loader:
        for i in range(x_b.size(0)):
            lemma  = lemma_list[lid_b[i].item()]
            logits = model(x_b[i:i+1], pos_b[i:i+1], lemma)
            pred   = logits.argmax(-1).item()
            lemma_correct[lemma] += int(pred == y_b[i].item())
            lemma_total[lemma]   += 1

for lemma in lemma_list:
    if lemma_total[lemma] > 0:
        acc = 100.0 * lemma_correct[lemma] / lemma_total[lemma]
        n   = len(lemma_sense_vocab[lemma])
        print(f"  {lemma:<12} {acc:5.1f}%  "
              f"({lemma_correct[lemma]}/{lemma_total[lemma]} correct, {n} senses)")

# ── STEP 15: PREDICT ─────────────────────────────────────────────
def predict(sentence: str, target_word: str):
    model.eval()
    tokens      = sentence.lower().split()
    target_word = target_word.lower()

    if target_word not in tokens:
        print(f"  '{target_word}' not found in sentence."); return
    if target_word not in multi_sense_lemmas:
        print(f"  '{target_word}' not in vocabulary."); return

    tgt_idx = tokens.index(target_word)
    ids     = [word_vocab.get(w, 1) for w in tokens]
    if len(ids) > MAX_LEN:
        start   = max(0, tgt_idx - MAX_LEN // 2)
        ids     = ids[start : start + MAX_LEN]
        tgt_idx = tgt_idx - start
    tgt_idx = min(tgt_idx, len(ids) - 1)
    ids     = ids + [0] * (MAX_LEN - len(ids))

    x_t   = torch.tensor([ids])
    pos_t = torch.tensor([tgt_idx])
    with torch.no_grad():
        logits = model(x_t, pos_t, target_word)
        probs  = torch.softmax(logits[0], dim=-1)

    id_to_sense = {v: k for k, v in lemma_sense_vocab[target_word].items()}
    n           = len(id_to_sense)

    print(f"\n  Sentence : {sentence}")
    print(f"  Target   : '{target_word}'")
    print(f"  {'-'*62}")
    topk = torch.topk(probs[:n], min(3, n))
    for rank, (idx, score) in enumerate(zip(topk.indices, topk.values), 1):
        sense = id_to_sense[idx.item()]
        defn  = sense_definitions.get(sense, sense)
        bar   = "█" * int(score.item() * 24)
        print(f"  #{rank}  {score.item():.2f}  {bar}")
        print(f"       {sense}")
        print(f"       {defn[:72]}")

# ── STEP 16: TEST ────────────────────────────────────────────────
print("\n" + "="*64)
print("TEST PREDICTIONS")
print("="*64)

predict("the bank near the river flooded last night",  "bank")
predict("i deposited money in the bank yesterday",     "bank")
predict("he hit the ball with the baseball bat",       "bat")
predict("the bat flew out of the cave at dusk",        "bat")
predict("the plant grew tall in the sunlight",         "plant")
predict("the factory is a large industrial plant",     "plant")
predict("the spring water was cold and clear",         "spring")
predict("the metal spring bounced back quickly",       "spring")
predict("i need to file my taxes before april",        "file")
predict("the file contained all the documents",        "file")
predict("the rock band played all night",              "rock")
predict("she climbed up the rock face carefully",      "rock")

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


cp: cannot stat 'kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory
Dataset URL: https://www.kaggle.com/datasets/nltkdata/semcor-corpus
License(s): other
semcor-corpus.zip: Skipping, found more recently modified local copy (use --force to force download)
Archive:  semcor-corpus.zip
  inflating: README                  
  inflating: semcor/semcor/INSTALL   
  inflating: semcor/semcor/LICENSE   
  inflating: semcor/semcor/README    
  inflating: semcor/semcor/brown1/tagfiles/br-a01.xml  
  inflating: semcor/semcor/brown1/tagfiles/br-a02.xml  
  inflating: semcor/semcor/brown1/tagfiles/br-a11.xml  
  inflating: semcor/semcor/brown1/tagfiles/br-a12.xml  
  inflating: semcor/semcor/brown1/tagfiles/br-a13.xml  
  inflating: semcor/semcor/brown1/tagfiles/br-a14.xml  
  inflating: semcor/semcor/brown1/tagfiles/br-a15.xml  
  inflating: semcor/semcor/brown1/tagfiles/br-b13.xml  
  inflating: semcor/semcor/brown1/tagfiles/br-b20.xm

/tmp/ipykernel_4603/2838646226.py:306: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder  = nn.TransformerEncoder(enc_layer, num_layers=n_layers)


Epoch   1  |  Loss: 203.785  |  Train: 28.2%  |  Val: 43.3%  |  Best: 43.3%  |  Patience: 0/20
Epoch  10  |  Loss: 148.048  |  Train: 54.6%  |  Val: 47.3%  |  Best: 47.5%  |  Patience: 6/20
Epoch  20  |  Loss: 123.197  |  Train: 70.0%  |  Val: 47.7%  |  Best: 49.5%  |  Patience: 3/20
Epoch  30  |  Loss: 106.739  |  Train: 79.9%  |  Val: 49.7%  |  Best: 50.3%  |  Patience: 8/20
Epoch  40  |  Loss: 95.176  |  Train: 88.0%  |  Val: 49.5%  |  Best: 50.3%  |  Patience: 18/20

Early stop at epoch 42  (best val: 50.3%)

Loaded best model — val acc: 50.3%

Per-lemma validation accuracy:
  arm           83.3%  (10/12 correct, 6 senses)
  bank          66.7%  (4/6 correct, 6 senses)
  bark         100.0%  (1/1 correct, 3 senses)
  bass           0.0%  (0/1 correct, 2 senses)
  bat           50.0%  (2/4 correct, 5 senses)
  bear          45.5%  (5/11 correct, 10 senses)
  board         44.4%  (4/9 correct, 6 senses)
  bow           33.3%  (1/3 correct, 7 senses)
  face          57.5%  (23/40 corr